# 1. Set up

In [1]:
!pip install pandas
!pip install polars
!pip install pyarrow

# 2. Import necessary libraries

In [2]:
import pandas as pd
import polars as pl
import numpy as np

import category_encoders as ce

from sklearn.model_selection import RepeatedKFold
from sklearn.ensemble import RandomForestClassifier

from collections import defaultdict

# 3. Define global variables

we will be using march as the training month and april as the validation month

In [3]:
INPUT_EVENTOS_FRAUDE_PATH = "../data/inputs/consumo_eventos/eventos_fraude_31_05_2025.csv"
INPUT_EVENTOS_CONTROL_PATH = "../data/inputs/consumo_eventos/eventos_no_fraude_31_05_2025.csv"
INPUT_CONSUMOS_FRAUDE_PATH = "../data/inputs/consumo_eventos/consumos_fraude_31_05_2025.csv" 
INPUT_CONSUMOS_CONTROL_PATH = "../data/inputs/consumo_eventos/consumos_no_fraude_31_05_2025.csv"
# INPUT_FRAUDES_PATH = "../data/inputs/consumo_eventos/fraudes_al_31_05_2025_fecha_fin.csv"


In [4]:
INPUT_EVENTOS_FRAUDE_PATH_ABRIL = "../data/inputs/consumo_eventos/eventos_fraude_30_06_2025.csv"
INPUT_EVENTOS_CONTROL_PATH_ABRIL = "../data/inputs/consumo_eventos/eventos_no_fraude_al_30_06_2025.csv"
INPUT_CONSUMOS_FRAUDE_PATH_ABRIL = "../data/inputs/consumo_eventos/consumos_fraude_30_06_2025.csv" 
INPUT_CONSUMOS_CONTROL_PATH_ABRIL = "../data/inputs/consumo_eventos/consumos_no_fraude_30_06_2025.csv" 
# INPUT_FRAUDES_PATH_ABRIL = "../data/inputs/consumo_eventos/fraudes_al_30_06_2025_fecha_fin.csv"

# INPUT_CONSUMOS_PATH_TEST = "../data/inputs/consumos_al_2025-09-24.csv"
# INPUT_EVENTOS_PATH_TEST = "../data/inputs/eventos_al_2025-09-24.csv"

# INPUT_GRID_CONTADORES_CONTROL_PATH_ABRIL = "../data/inputs/grid_contadores_no_fraude_30_04_2025.csv"
# INPUT_GRID_CONTADORES_FRAUDE_PATH_ABRIL = "../data/inputs/grid_contadores_fraude_30_04_2025.csv"

In [5]:
OUTPUT_X_PATH = "../data/output_data/data_training_may.csv"
OUTPUT_X_EVAL_PATH = "../data/output_data/data_validation_june.csv"
OUTPUT_Y_EVAL_PATH = "../data/output_data/data_validation_june_y.csv"
OUTPUT_Y_PATH = "../data/output_data/data_training_may_y.csv"

OUTPUT_FINAL_DF_PATH  = "../data/output_data/final_df_may.csv"
OUTPUT_FINAL_DF_EVAL_PATH  = "../data/output_data/final_df_eval_june.csv"

In [6]:
train_losses = list()
target_col = "target"

Let's define some colors for the viz we will be doing

In [7]:
ufd_orange = "#f26122"
ufd_blue = "#003865"
ufd_gray = "#cccccc"

# 4. Functions

In [8]:
def crear_pivot_detallado(df: pd.DataFrame, 
                          columnas_inspecciones: list,
                          columnas_eventos: list) -> pd.DataFrame:
    """
    Crea un pivot manteniendo toda la información de conteos por combinación et-c
    """
    # Primero, crear una columna que combine et y c para identificar cada tipo de evento
    df_trabajo = df.copy()
    df_trabajo['tipo_evento'] = df_trabajo['et'].astype(str) + '_' + df_trabajo['c'].astype(str)
   
    # Crear el diccionario de agregación base
    agg_dict = {}
   
    # Para las columnas de inspecciones: tomar el primer valor (son iguales para cada grupo)
    for col in columnas_inspecciones:
        agg_dict[col] = 'first'
   
    # Para las columnas de eventos: necesitamos tratamiento especial
    # Las agregaremos después del groupby usando pivot
   
    # Primero hacer el groupby básico para las columnas de inspecciones
    resultado_base = df_trabajo.groupby(['cups_sgc', 'cnt_id', 'target']).agg(agg_dict).reset_index()
   
    # Ahora crear el pivot para las columnas de eventos manteniendo la granularidad
    eventos_pivot_list = []
   
    for col_evento in columnas_eventos:
        # Crear un pivot para cada columna de eventos
        pivot_temp = df_trabajo.pivot_table(
            index=['cups_sgc', 'cnt_id', 'target'],
            columns='tipo_evento',
            values=col_evento,
            aggfunc='sum',
            fill_value=0
        ).reset_index()
       
        # Renombrar las columnas para incluir el nombre de la columna original
        pivot_temp.columns = ['cups_sgc', 'cnt_id', 'target'] + [f"{col_evento}_{col}" for col in pivot_temp.columns[3:]]
       
        eventos_pivot_list.append(pivot_temp)
   
    # Unir todos los pivots de eventos
    resultado_eventos = eventos_pivot_list[0]
    for pivot in eventos_pivot_list[1:]:
        resultado_eventos = resultado_eventos.merge(
            pivot,
            on=['cups_sgc', 'cnt_id', 'target'],
            how='outer'
        )
   
    # Unir con el resultado base (inspecciones)
    resultado_final = resultado_base.merge(
        resultado_eventos,
        on=['cups_sgc', 'cnt_id', 'target'],
        how='outer'
    )
   
    return resultado_final

# 5. Code

## 5.1. Data processing

We will load the data, both the training and the validation data

In [9]:
eventos_fraude = pl.read_csv(INPUT_EVENTOS_FRAUDE_PATH)
eventos_control = pl.read_csv(INPUT_EVENTOS_CONTROL_PATH)
consumos_fraude = pl.read_csv(INPUT_CONSUMOS_FRAUDE_PATH).drop(["tuvo_fraude_previo", "tuvo_fraude_reciente", "estado_fraude"])
consumos_control = pl.read_csv(INPUT_CONSUMOS_CONTROL_PATH).drop(["tuvo_fraude_previo", "tuvo_fraude_reciente", "estado_fraude"])
# fraudes = pd.read_csv(INPUT_FRAUDES_PATH).rename(columns={"cups": "cups_sgc", "fraudes_en_90_dias": "target"})

# sólo hay que pivotar la tabla de eventos para después unirla con las de consumos
eventos_fraude = eventos_fraude.with_columns(pl.lit(1).alias("target"))
eventos_control = eventos_control.with_columns(pl.lit(0).alias("target"))

eventos = pl.concat([eventos_control, eventos_fraude])
consumos = pl.concat([consumos_control, consumos_fraude])

In [10]:
eventos_fraude_abril = pl.read_csv(INPUT_EVENTOS_FRAUDE_PATH_ABRIL)
eventos_control_abril = pl.read_csv(INPUT_EVENTOS_CONTROL_PATH_ABRIL)
consumos_fraude_abril = pl.read_csv(INPUT_CONSUMOS_FRAUDE_PATH_ABRIL).drop(["tuvo_fraude_previo", "tuvo_fraude_reciente", "estado_fraude"])
consumos_control_abril = pl.read_csv(INPUT_CONSUMOS_CONTROL_PATH_ABRIL).drop(["tuvo_fraude_previo", "tuvo_fraude_reciente", "estado_fraude"])
# fraudes_abril = pd.read_csv(INPUT_FRAUDES_PATH_ABRIL).rename(columns={"cups": "cups_sgc", "fraudes_en_90_dias": "target"})

# # sólo hay que pivotar la tabla de eventos para después unirla con las de consumos
eventos_fraude_abril = eventos_fraude_abril.with_columns(pl.lit(1).alias("target"))
eventos_control_abril = eventos_control_abril.with_columns(pl.lit(0).alias("target"))


eventos_abril = pl.concat([eventos_control_abril, eventos_fraude_abril])
consumos_abril = pl.concat([consumos_control_abril, consumos_fraude_abril])


# eventos_test = pl.read_csv(INPUT_EVENTOS_PATH_TEST)
# consumos_test = pl.read_csv(INPUT_CONSUMOS_PATH_TEST).drop(["tuvo_fraude_previo", "tuvo_fraude_reciente", "estado_fraude"])


In [11]:
import random
random.seed(90)


In [12]:
eventos_test = eventos_abril
consumos_test = consumos_abril


In [13]:
# Extraemos las columnas numéricas de la tabla eventos
columnas_eventos = [col for col in eventos.columns if col not in ['cups_sgc', 'cnt_id', 'et', 'c'] and "eventos" in col]
columnas_inspecciones = [col for col in eventos.columns if col not in ['cups_sgc', 'cnt_id', 'et', 'c'] and "inspecciones" in col]

# Debemos rellenar los nulos de las columnas 'c' y 'et' antes de crear 'tipo_evento'
eventos = eventos.with_columns([
    pl.col("c").fill_null("NA").cast(pl.Utf8),
    pl.col("et").fill_null("NA").cast(pl.Utf8)
])
eventos_test = eventos_test.with_columns([
    pl.col("c").fill_null("NA").cast(pl.Utf8),
    pl.col("et").fill_null("NA").cast(pl.Utf8)
])

# Creación de la columna tipo_evento
eventos = eventos.with_columns(
    (pl.col("et") + "_" + pl.col("c")).alias("tipo_evento")
)
eventos_test = eventos_test.with_columns(
    (pl.col("et") + "_" + pl.col("c")).alias("tipo_evento")
)

Convertimos los polars data frames a pandas df:

In [14]:
eventos_df_pandas = eventos.to_pandas()
consumos_pandas = consumos.to_pandas()

eventos_df_pandas_test = eventos_test.to_pandas()
consumos_pandas_test = consumos_test.to_pandas()

In [15]:
eventos.head(10)

cups_sgc,cnt_id,et,c,cnt_eventos_semana_actual,cnt_eventos_semana_pasada,cnt_eventos_misma_semana_anio_pasado,cnt_eventos_mes_actual,cnt_eventos_mes_pasado,cnt_eventos_mismo_mes_anio_pasado,cnt_eventos_trimestre_actual,cnt_eventos_trimestre_pasado,cnt_eventos_mismo_trimestre_anio_pasado,cnt_eventos_anio_actual,cnt_eventos_anio_pasado,cnt_inspecciones_semana_actual,cnt_inspecciones_semana_pasada,cnt_inspecciones_misma_semana_anio_pasado,cnt_inspecciones_mes_actual,cnt_inspecciones_mes_pasado,cnt_inspecciones_mismo_mes_anio_pasado,cnt_inspecciones_trimestre_actual,cnt_inspecciones_trimestre_pasado,cnt_inspecciones_mismo_trimestre_anio_pasado,cnt_inspecciones_anio_actual,cnt_inspecciones_anio_pasado,target,tipo_evento
str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i32,str
"""ES0022000001000050VC1P""","""SAG0185746545""","""2""","""6""",0,0,0,0,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,"""2_6"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""1""","""4""",0,0,0,0,1,0,1,0,1,1,2,0,0,0,0,0,0,0,0,0,0,0,0,"""1_4"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""1""","""25""",0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,"""1_25"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""3""","""22""",0,0,0,0,0,0,0,0,1,0,4,0,0,0,0,0,0,0,0,0,0,0,0,"""3_22"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""4""","""6""",7,7,7,30,28,29,58,88,46,332,46,0,0,0,0,0,0,0,0,0,0,0,0,"""4_6"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""6""","""NA""",46,50,50,222,216,214,438,656,402,2496,2036,0,0,0,0,0,0,0,0,0,0,0,0,"""6_NA"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""3""","""18""",46,44,0,207,151,0,358,258,3,632,11,0,0,0,0,0,0,0,0,0,0,0,0,"""3_18"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""2""","""14""",0,0,0,0,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,"""2_14"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""1""","""21""",0,0,0,0,1,0,1,0,1,1,2,0,0,0,0,0,0,0,0,0,0,0,0,"""1_21"""


In [16]:
eventos_test.head(10)

cups_sgc,cnt_id,et,c,cnt_eventos_semana_actual,cnt_eventos_semana_pasada,cnt_eventos_misma_semana_anio_pasado,cnt_eventos_mes_actual,cnt_eventos_mes_pasado,cnt_eventos_mismo_mes_anio_pasado,cnt_eventos_trimestre_actual,cnt_eventos_trimestre_pasado,cnt_eventos_mismo_trimestre_anio_pasado,cnt_eventos_anio_actual,cnt_eventos_anio_pasado,cnt_inspecciones_semana_actual,cnt_inspecciones_semana_pasada,cnt_inspecciones_misma_semana_anio_pasado,cnt_inspecciones_mes_actual,cnt_inspecciones_mes_pasado,cnt_inspecciones_mismo_mes_anio_pasado,cnt_inspecciones_trimestre_actual,cnt_inspecciones_trimestre_pasado,cnt_inspecciones_mismo_trimestre_anio_pasado,cnt_inspecciones_anio_actual,cnt_inspecciones_anio_pasado,target,tipo_evento
str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i32,str
"""ES0022000001000050VC1P""","""SAG0185746545""","""6""","""NA""",50,50,52,220,226,222,662,656,624,2498,2102,0,0,0,0,0,0,0,0,0,0,0,0,"""6_NA"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""1""","""25""",0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,"""1_25"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""1""","""24""",0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,"""1_24"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""2""","""6""",0,0,0,0,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,"""2_6"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""1""","""21""",1,0,0,1,0,0,2,0,1,2,2,0,0,0,0,0,0,0,0,0,0,0,0,"""1_21"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""3""","""14""",0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,"""3_14"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""1""","""44""",0,0,0,1,1,1,3,3,3,12,12,0,0,0,0,0,0,0,0,0,0,0,0,"""1_44"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""2""","""14""",0,0,0,0,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,"""2_14"""
"""ES0022000001000050VC1P""","""SAG0185746545""","""1""","""3""",2,0,0,2,1,1,4,1,2,10,14,0,0,0,0,0,0,0,0,0,0,0,0,"""1_3"""


Vamos a pivotar la tabla de eventos para tener un único valor de cups por cada fila:

In [17]:
# Ejecutar la función
print("Creando pivot detallado...")
eventos_pivotado_detallado = crear_pivot_detallado(eventos_df_pandas, columnas_inspecciones, columnas_eventos)
eventos_pivotado_detallado_test = crear_pivot_detallado(eventos_df_pandas_test, columnas_inspecciones, columnas_eventos)

Creando pivot detallado...


In [18]:
eventos_pivotado_detallado.head(10)

,cups_sgc,cnt_id,target,cnt_inspecciones_semana_actual,cnt_inspecciones_semana_pasada,cnt_inspecciones_misma_semana_anio_pasado,cnt_inspecciones_mes_actual,cnt_inspecciones_mes_pasado,cnt_inspecciones_mismo_mes_anio_pasado,cnt_inspecciones_trimestre_actual,...,cnt_eventos_anio_pasado_4_3,cnt_eventos_anio_pasado_4_4,cnt_eventos_anio_pasado_4_5,cnt_eventos_anio_pasado_4_6,cnt_eventos_anio_pasado_4_7,cnt_eventos_anio_pasado_4_8,cnt_eventos_anio_pasado_4_9,cnt_eventos_anio_pasado_5_NA,cnt_eventos_anio_pasado_6_NA,cnt_eventos_anio_pasado_7_NA
0,ES0022000001000008QR1P,SAG0185769835,1,0,0,0,0,28,0,28,...,0,0,0,63,0,0,0,0,1378,1
1,ES0022000001000050VC1P,SAG0185746545,0,0,0,0,0,0,0,0,...,0,0,0,46,0,0,0,0,2036,0
2,ES0022000001000051VK1P,ZIV0048682250,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,20,40
3,ES0022000001000070HV1P,SAG0185746689,0,0,0,0,0,0,0,0,...,0,0,0,248,0,0,0,11,2106,5
4,ES0022000001000071HH1P,ZIV0048687588,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,2,1910,0
5,ES0022000001000072HL1P,SAG0175697550,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,510,1910,0
6,ES0022000001000073HC1P,ZIV0049764177,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1919,0
7,ES0022000001000074HK1P,ZIV0049764171,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,15,1901,0
8,ES0022000001000075HE1P,SAG0175697713,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,6,1955,0
9,ES0022000001000091LS1P,ZIV0048687590,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,4,1905,0


In [19]:
len(eventos_pivotado_detallado)

205373

In [20]:
eventos_pivotado_detallado_test.head(10)

,cups_sgc,cnt_id,target,cnt_inspecciones_semana_actual,cnt_inspecciones_semana_pasada,cnt_inspecciones_misma_semana_anio_pasado,cnt_inspecciones_mes_actual,cnt_inspecciones_mes_pasado,cnt_inspecciones_mismo_mes_anio_pasado,cnt_inspecciones_trimestre_actual,...,cnt_eventos_anio_pasado_4_3,cnt_eventos_anio_pasado_4_4,cnt_eventos_anio_pasado_4_5,cnt_eventos_anio_pasado_4_6,cnt_eventos_anio_pasado_4_7,cnt_eventos_anio_pasado_4_8,cnt_eventos_anio_pasado_4_9,cnt_eventos_anio_pasado_5_NA,cnt_eventos_anio_pasado_6_NA,cnt_eventos_anio_pasado_7_NA
0,ES0022000001000008QR1P,SAG0185769835,1,0,0,0,0,0,0,28,...,0,0,0,53,0,0,0,0,1368,1
1,ES0022000001000050VC1P,SAG0185746545,0,0,0,0,0,0,0,0,...,0,0,0,75,0,0,0,0,2102,0
2,ES0022000001000051VK1P,ZIV0048682250,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,168,40
3,ES0022000001000070HV1P,SAG0185746689,0,0,0,0,0,0,0,0,...,0,0,0,247,0,0,0,11,2174,6
4,ES0022000001000071HH1P,ZIV0048687588,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,2,1918,0
5,ES0022000001000072HL1P,SAG0175697550,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,544,1920,0
6,ES0022000001000073HC1P,ZIV0049764177,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1923,0
7,ES0022000001000074HK1P,ZIV0049764171,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,14,1909,0
8,ES0022000001000075HE1P,SAG0175697713,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,6,1963,0
9,ES0022000001000091LS1P,ZIV0048687590,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,4,1915,0


In [21]:
len(eventos_pivotado_detallado_test)

205422

In [22]:
final_df = eventos_pivotado_detallado.merge(
    consumos.to_pandas(), on=["cups_sgc", "cnt_id"], how="outer"
)#.merge(fraudes, on=["cups_sgc"], how="left")


final_df.drop_duplicates(inplace=True)

final_df.head(10)

,cups_sgc,cnt_id,target,cnt_inspecciones_semana_actual,cnt_inspecciones_semana_pasada,cnt_inspecciones_misma_semana_anio_pasado,cnt_inspecciones_mes_actual,cnt_inspecciones_mes_pasado,cnt_inspecciones_mismo_mes_anio_pasado,cnt_inspecciones_trimestre_actual,...,porcentaje_calidad_datos_trimestre_actual_correctos,porcentaje_calidad_datos_trimestre_actual_incorrectos,porcentaje_calidad_datos_trimestre_pasado_correctos,porcentaje_calidad_datos_trimestre_pasado_incorrectos,porcentaje_calidad_datos_mismo_trimestre_anio_pasado_correctos,porcentaje_calidad_datos_mismo_trimestre_anio_pasado_incorrectos,porcentaje_calidad_datos_anio_actual_correctos,porcentaje_calidad_datos_anio_actual_incorrectos,porcentaje_calidad_datos_anio_anterior_correctos,porcentaje_calidad_datos_anio_anterior_incorrectos
0,ES0022000001000008QR1P,SAG0185769835,1.0,0.0,0.0,0.0,0.0,28.0,0.0,28.0,...,98.70,1.30,98.84,0.0,100.00,0.00,97.31,0.22,99.44,0.00
1,ES0022000001000050VC1P,SAG0185746545,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.63,1.37,105.51,0.0,99.86,0.14,98.13,0.23,101.84,0.05
2,ES0022000001000051VK1P,ZIV0048682250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,75.75,1.37,105.51,0.0,6.56,0.00,85.80,0.23,5.75,0.00
3,ES0022000001000052VE1P,ZIV0048682245,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,98.91,1.09,112.18,0.0,100.00,0.00,100.63,0.18,99.72,0.00
4,ES0022000001000070HV1P,SAG0185746689,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.77,1.23,102.18,0.0,98.36,0.00,98.40,0.21,98.61,0.00
5,ES0022000001000071HH1P,ZIV0048687588,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,97.40,0.96,105.51,0.0,96.72,0.00,98.46,0.16,98.62,0.00
6,ES0022000001000072HL1P,SAG0175697550,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,97.40,0.96,105.51,0.0,95.08,0.00,98.18,0.16,97.53,0.00
7,ES0022000001000073HC1P,ZIV0049764177,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,97.40,0.96,105.51,0.0,96.72,0.00,98.46,0.16,98.62,0.00
8,ES0022000001000074HK1P,ZIV0049764171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,97.40,0.96,105.51,0.0,96.72,0.00,98.46,0.16,98.62,0.00
9,ES0022000001000075HE1P,SAG0175697713,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,97.40,0.96,105.51,0.0,96.72,0.00,97.36,0.16,96.98,0.00


In [23]:
# final_df_eval = eventos_pivotado_detallado_test.merge(
#     consumos_pandas_test, on=["cups_sgc", "cnt_id"], how="outer"
# ).merge(fraudes_abril, on=["cups_sgc"], how="left")

final_df_eval = eventos_pivotado_detallado_test.merge(
    consumos_pandas_test, on=["cups_sgc", "cnt_id"], how="outer"
)

final_df_eval.drop_duplicates(inplace=True)

final_df_eval.head(10)

,cups_sgc,cnt_id,target,cnt_inspecciones_semana_actual,cnt_inspecciones_semana_pasada,cnt_inspecciones_misma_semana_anio_pasado,cnt_inspecciones_mes_actual,cnt_inspecciones_mes_pasado,cnt_inspecciones_mismo_mes_anio_pasado,cnt_inspecciones_trimestre_actual,...,porcentaje_calidad_datos_trimestre_actual_correctos,porcentaje_calidad_datos_trimestre_actual_incorrectos,porcentaje_calidad_datos_trimestre_pasado_correctos,porcentaje_calidad_datos_trimestre_pasado_incorrectos,porcentaje_calidad_datos_mismo_trimestre_anio_pasado_correctos,porcentaje_calidad_datos_mismo_trimestre_anio_pasado_incorrectos,porcentaje_calidad_datos_anio_actual_correctos,porcentaje_calidad_datos_anio_actual_incorrectos,porcentaje_calidad_datos_anio_anterior_correctos,porcentaje_calidad_datos_anio_anterior_incorrectos
0,ES0022000001000008QR1P,SAG0185769835,1.0,0.0,0.0,0.0,0.0,0.0,0.0,28.0,...,99.13,0.87,98.84,0.0,100.00,0.00,97.31,0.22,99.44,0.00
1,ES0022000001000050VC1P,SAG0185746545,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.99,1.01,105.51,0.0,99.91,0.09,98.11,0.25,101.84,0.05
2,ES0022000001000051VK1P,ZIV0048682250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,83.70,0.92,105.51,0.0,48.35,0.00,83.05,0.23,16.68,0.00
3,ES0022000001000052VE1P,ZIV0048682245,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,99.27,0.73,112.18,0.0,100.00,0.00,100.63,0.18,102.17,0.00
4,ES0022000001000070HV1P,SAG0185746689,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,99.18,0.82,102.18,0.0,97.76,0.00,98.69,0.21,98.33,0.00
5,ES0022000001000071HH1P,ZIV0048687588,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.26,0.64,105.51,0.0,96.70,0.00,98.73,0.16,98.35,0.00
6,ES0022000001000072HL1P,SAG0175697550,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.26,0.64,105.51,0.0,94.51,0.00,98.73,0.16,96.98,0.00
7,ES0022000001000073HC1P,ZIV0049764177,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.26,0.64,105.51,0.0,96.70,0.00,98.73,0.16,98.35,0.00
8,ES0022000001000074HK1P,ZIV0049764171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.26,0.64,105.51,0.0,96.70,0.00,98.73,0.16,98.35,0.00
9,ES0022000001000075HE1P,SAG0175697713,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.26,0.64,105.51,0.0,95.60,0.00,97.91,0.16,96.71,0.00


### 5.1.1. Data at cups level

We are going to group all the data at cups level

### 5.1.2. Constant columns

We will delete those columns that have a unique value, thus, constant columns

In [24]:
constant_cols = []

for col in final_df.columns:
    if final_df[col].nunique() == 1 and col not in ['cups_sgc', 'cnt_id', target_col]:
        constant_cols.append(col)
        
        
final_df.drop(columns=constant_cols, inplace=True)

In [25]:
constant_cols_eval = []

for col in final_df_eval.columns:
    if final_df_eval[col].nunique() == 1 and col not in ['cups_sgc', 'cnt_id', target_col]:
        constant_cols.append(col)
        
        
final_df_eval.drop(columns=constant_cols_eval, inplace=True)

### 5.1.3. Duplicated rows

Are there any duplicated rows in the data?

In [26]:
final_df[[col for col in final_df if col not in ['cups_sgc', 'cnt_id']]].duplicated().sum()

np.int64(3)

We observe that there are some little duplicated rows, we are going to delete them

In [27]:
final_df = final_df[~final_df[[col for col in final_df if col not in ['cups_sgc', 'cnt_id']]].duplicated()]

final_df_eval = final_df_eval[~final_df_eval[[col for col in final_df_eval if col not in ['cups_sgc', 'cnt_id']]].duplicated()]

In [28]:
final_df.head(5)

,cups_sgc,cnt_id,target,cnt_inspecciones_semana_actual,cnt_inspecciones_semana_pasada,cnt_inspecciones_misma_semana_anio_pasado,cnt_inspecciones_mes_actual,cnt_inspecciones_mes_pasado,cnt_inspecciones_mismo_mes_anio_pasado,cnt_inspecciones_trimestre_actual,...,porcentaje_calidad_datos_trimestre_actual_correctos,porcentaje_calidad_datos_trimestre_actual_incorrectos,porcentaje_calidad_datos_trimestre_pasado_correctos,porcentaje_calidad_datos_trimestre_pasado_incorrectos,porcentaje_calidad_datos_mismo_trimestre_anio_pasado_correctos,porcentaje_calidad_datos_mismo_trimestre_anio_pasado_incorrectos,porcentaje_calidad_datos_anio_actual_correctos,porcentaje_calidad_datos_anio_actual_incorrectos,porcentaje_calidad_datos_anio_anterior_correctos,porcentaje_calidad_datos_anio_anterior_incorrectos
0,ES0022000001000008QR1P,SAG0185769835,1.0,0.0,0.0,0.0,0.0,28.0,0.0,28.0,...,98.70,1.30,98.84,0.0,100.00,0.00,97.31,0.22,99.44,0.00
1,ES0022000001000050VC1P,SAG0185746545,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.63,1.37,105.51,0.0,99.86,0.14,98.13,0.23,101.84,0.05
2,ES0022000001000051VK1P,ZIV0048682250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,75.75,1.37,105.51,0.0,6.56,0.00,85.80,0.23,5.75,0.00
3,ES0022000001000052VE1P,ZIV0048682245,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,98.91,1.09,112.18,0.0,100.00,0.00,100.63,0.18,99.72,0.00
4,ES0022000001000070HV1P,SAG0185746689,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.77,1.23,102.18,0.0,98.36,0.00,98.40,0.21,98.61,0.00


### 5.1.3. Variables selection

In [29]:
# final_df.drop(columns=[col for col in final_df.columns if "porcentaje" in col], inplace=True)
# final_df.drop(columns=[col for col in final_df.columns if "consumo_zero" in col], inplace=True)

We extract the categorical and continuous columns as well as we define the X and y sets

In [30]:
cat_cols = final_df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [col for col in cat_cols if col not in ['cups_sgc', 'cnt_id', target_col]]

X = final_df.drop(columns=[target_col, 'cups_sgc', 'cnt_id'], errors='ignore')
y = final_df[target_col]

X_eval = final_df_eval.drop(columns=[target_col, 'cups_sgc', 'cnt_id'], errors='ignore')
y_eval = final_df_eval[target_col]

In [32]:
X = X.apply(pd.to_numeric, errors='coerce', )

# Opción 1: rellenar NaN con 0
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(-1)

# Clip very large/small values to avoid dtype errors
X = X.clip(lower=-1e6, upper=1e6)

X.head(5)

,cnt_inspecciones_semana_actual,cnt_inspecciones_semana_pasada,cnt_inspecciones_misma_semana_anio_pasado,cnt_inspecciones_mes_actual,cnt_inspecciones_mes_pasado,cnt_inspecciones_mismo_mes_anio_pasado,cnt_inspecciones_trimestre_actual,cnt_inspecciones_trimestre_pasado,cnt_inspecciones_mismo_trimestre_anio_pasado,cnt_inspecciones_anio_actual,...,porcentaje_calidad_datos_trimestre_actual_correctos,porcentaje_calidad_datos_trimestre_actual_incorrectos,porcentaje_calidad_datos_trimestre_pasado_correctos,porcentaje_calidad_datos_trimestre_pasado_incorrectos,porcentaje_calidad_datos_mismo_trimestre_anio_pasado_correctos,porcentaje_calidad_datos_mismo_trimestre_anio_pasado_incorrectos,porcentaje_calidad_datos_anio_actual_correctos,porcentaje_calidad_datos_anio_actual_incorrectos,porcentaje_calidad_datos_anio_anterior_correctos,porcentaje_calidad_datos_anio_anterior_incorrectos
0,0.0,0.0,0.0,0.0,28.0,0.0,28.0,0.0,0.0,28.0,...,98.70,1.30,98.84,0.0,100.00,0.00,97.31,0.22,99.44,0.00
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.63,1.37,105.51,0.0,99.86,0.14,98.13,0.23,101.84,0.05
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,75.75,1.37,105.51,0.0,6.56,0.00,85.80,0.23,5.75,0.00
3,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,...,98.91,1.09,112.18,0.0,100.00,0.00,100.63,0.18,99.72,0.00
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,98.77,1.23,102.18,0.0,98.36,0.00,98.40,0.21,98.61,0.00


In [33]:
X_eval = X_eval.apply(pd.to_numeric, errors='coerce')

# Opción 1: rellenar NaN con 0
X_eval = X_eval.replace([np.inf, -np.inf], np.nan)
X_eval = X_eval.fillna(-1)

# Clip very large/small values to avoid dtype errors
X_eval = X_eval.clip(lower=-1e6, upper=1e6)

In [34]:
modelo_rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)

# Configura la validación cruzada.
# RepeatedKFold es una excelente opción.
# n_splits=5 divide tus datos en 5 "folds" o subconjuntos.
# n_repeats=2 repite todo el proceso 2 veces, lo que te da un total de 10 modelos
# y un ranking de importancia muy robusto.
cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

# Un diccionario para guardar las importancias de cada columna
importances = defaultdict(list)

y.fillna(0, inplace=True)

# Bucle sobre los splits de validación cruzada
for train_index, val_index in cv.split(X, y):
    X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

    # Entrena el modelo en el subconjunto de entrenamiento
    # cols_to_select = [col for col in X_train_fold if col not in ["cnt_id", "cups_sgc"]]
    # X_train_fold, X_val_fold = X_train_fold[cols_to_select], X_val_fold[cols_to_select]
    modelo_rf.fit(X_train_fold, y_train_fold)

    # Guarda las importancias de las características
    for i, col in enumerate(X.columns):
        importances[col].append(modelo_rf.feature_importances_[i])

# Calcular la media de las importancias para cada columna
feature_importances_mean = {col: np.mean(imps) for col, imps in importances.items()}

# Convertir a una Serie de pandas para ordenar fácilmente
importances_series = pd.Series(feature_importances_mean).sort_values(ascending=False)

# Seleccionar las 100 mejores características
top_100_features = importances_series.head(125).index.tolist()

In [35]:
X = X[top_100_features]
X_eval = X_eval[top_100_features]

## 5.2. Write the results

In [36]:
final_df.to_csv(OUTPUT_FINAL_DF_PATH, sep=";", index=False)
X.to_csv(OUTPUT_X_PATH, sep=";", index=False)
y.to_csv(OUTPUT_Y_PATH, sep=";", index=False)

final_df_eval.to_csv(OUTPUT_FINAL_DF_EVAL_PATH, sep=";", index=False)
X_eval.to_csv(OUTPUT_X_EVAL_PATH, sep=";", index=False)
y_eval.to_csv(OUTPUT_Y_EVAL_PATH, sep=";", index=False)